# Morphological Blemish Analysis
### Developer: Herman

Pure mathematical-morphology for mango ripeness assessment from skin blemishes.

### Imports

In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import morphology as skm, measure as skmeas
from scipy import ndimage as ndi
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Step 1: Fruit Mask (black-bg threshold)

In [ ]:
# cleaned_data: background removed -> pure black. Mask = non-black pixels.
def get_fruit_mask(img, thresh=30):
    return (img.sum(axis=2) > thresh).astype(np.uint8)

### Step 2: Mask Cleanup (opening -> closing)

In [ ]:
# opening removes specks, closing fills holes in the mask.
def clean_mask(mask, k=5):
    kernel = np.ones((k, k), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

### Step 3: Interior Restriction (erode mask)

In [ ]:
# erode mask -> skip fruit boundary ring (false blemish band from bg edge).
def erode_mask(mask, k=9):
    kernel = np.ones((k, k), np.uint8)
    return cv2.erode(mask, kernel)

### Step 4: Illumination Flattening (white top-hat)

In [ ]:
# white top-hat = img - opening(img): removes large-scale shading, keeps small details.
def flatten_illumination(gray, k=31):
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    return cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, se)

### Step 5: Denoise (grayscale opening/closing)

In [ ]:
# grayscale open+close with small SE: smooths noise without blurring blemishes.
def denoise_gray(gray, k=3):
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    gray = cv2.morphologyEx(gray, cv2.MORPH_OPEN, se)
    gray = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, se)
    return gray

### Step 6: Blemish Enhancement (black + white top-hat)

In [ ]:
# black top-hat = closing - img -> dark spots/lesions/bruises.
# white top-hat = img - opening -> bright/pale patches.
def enhance_blemishes(gray, k=15):
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    dark   = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, se)
    bright = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, se)
    return dark, bright

### Step 7: Binarize Blemish Map

In [ ]:
def binarize(resp, thresh=40):
    _, bw = cv2.threshold(resp, thresh, 255, cv2.THRESH_BINARY)
    return bw

### Step 8: Blemish Cleanup (open/close, small objects, fill holes)

In [ ]:
def cleanup_blemishes(bw, min_area=100):
    kernel = np.ones((3, 3), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel)
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel)
    bw = skm.remove_small_objects(bw.astype(bool), min_size=min_area)
    bw = ndi.binary_fill_holes(bw)
    return (bw * 255).astype(np.uint8)

### Step 9: Split Touching Spots (marker-based watershed)

In [ ]:
# markers from distance transform; watershed on morphological gradient separates touching blemishes.
def watershed_split(gray, bw):
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    grad = cv2.morphologyEx(gray, cv2.MORPH_GRADIENT, se)
    dist = cv2.distanceTransform(bw, cv2.DIST_L2, 5)
    _, seed = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
    markers = cv2.connectedComponents(seed.astype(np.uint8))[1]
    markers = cv2.watershed(cv2.cvtColor(grad, cv2.COLOR_GRAY2BGR), markers)
    return markers  # -1 = boundary, 1..N = blemish regions

### Step 10: Feature Extraction (regionprops + skeletonize)

In [ ]:
# blemish statistics -> prediction features. skeletonize gives elongation.
def extract_features(gray, bw, markers, fruit_area):
    labels = markers.copy()
    labels[labels < 0] = 0  # drop watershed boundary
    props = [p for p in skmeas.regionprops(labels, intensity_image=gray) if p.label > 0]

    feats = {'n_blemishes': 0, 'blemish_area': 0, 'area_ratio': 0.0,
             'mean_darkness': 0.0, 'mean_brightness': 0.0,
             'mean_circularity': 0.0, 'mean_solidity': 0.0,
             'mean_eccentricity': 0.0, 'skeleton_length': 0}
    if not props or fruit_area == 0:
        return feats

    areas = [p.area for p in props]
    feats['n_blemishes'] = len(props)
    feats['blemish_area'] = int(sum(areas))
    feats['area_ratio'] = feats['blemish_area'] / fruit_area
    feats['mean_darkness']   = float(np.mean([255 - p.mean_intensity for p in props]))
    feats['mean_brightness'] = float(np.mean([p.mean_intensity for p in props]))
    per = [max(p.perimeter, 1e-6) for p in props]
    feats['mean_circularity']  = float(np.mean([4 * np.pi * a / pe**2 for a, pe in zip(areas, per)]))
    feats['mean_solidity']     = float(np.mean([p.solidity for p in props]))
    feats['mean_eccentricity'] = float(np.mean([p.eccentricity for p in props]))
    feats['skeleton_length']   = int(skm.skeletonize(bw > 0).sum())
    return feats

### Step 11: Full Morphology Pipeline (one image -> features)

In [ ]:
# chains steps 1-10; returns features + intermediates for visualization.
def morphology_pipeline(img):
    mask    = clean_mask(get_fruit_mask(img))     # 1-2
    interior = erode_mask(mask)                   # 3
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.bitwise_and(gray, gray, mask=interior)
    gray = flatten_illumination(gray)             # 4
    gray = denoise_gray(gray)                     # 5
    dark, bright = enhance_blemishes(gray)        # 6
    dark   = cv2.bitwise_and(dark, dark, mask=interior)
    bright = cv2.bitwise_and(bright, bright, mask=interior)
    bw = cv2.bitwise_or(cleanup_blemishes(binarize(dark)),   # 7-8
                        cleanup_blemishes(binarize(bright)))
    markers = watershed_split(gray, bw)           # 9
    feats = extract_features(gray, bw, markers, int(interior.sum()))  # 10
    vis = {'mask': mask, 'interior': interior, 'gray': gray,
           'dark': dark, 'bright': bright, 'bw': bw, 'markers': markers}
    return feats, vis

### Step 12: Inspect Pipeline on One Image per Class

In [ ]:
# quick sanity check: original, mask, blemish map, watershed labels.
fig, axes = plt.subplots(3, 5, figsize=(16, 9))
for r, cls in enumerate(['unripe', 'overripe', 'fully_ripe']):
    path = sorted(glob.glob(f'../cleaned_data/train/{cls}/*.jpg'))[0]
    img = cv2.imread(path)
    feats, vis = morphology_pipeline(img)
    imgs = [img, vis['mask'] * 255, vis['bw'], vis['markers'], img]
    for c, (title, im) in enumerate(zip(['orig', 'mask', 'blemish', 'watershed', 'overlay'], imgs)):
        axes[r, c].imshow(im, cmap='gray' if c in (1, 2, 3) else None)
        axes[r, c].set_title(f'{cls} | {title} | n={feats["n_blemishes"]}')
        axes[r, c].axis('off')
plt.tight_layout(); plt.show()

### Step 13: Build Feature Table (train split)

In [ ]:
ROOT = '../cleaned_data'
CLASSES = ['unripe', 'overripe', 'fully_ripe']

def build_features(split):
    rows = []
    for cls in CLASSES:
        for path in sorted(glob.glob(f'{ROOT}/{split}/{cls}/*.jpg')):
            feats, _ = morphology_pipeline(cv2.imread(path))
            feats['class'] = cls
            rows.append(feats)
    return pd.DataFrame(rows)

df_train = build_features('train')
print(df_train.shape)
df_train.head()

### Step 14: Train Classifier (Random Forest)

In [ ]:
FEAT_COLS = [c for c in df_train.columns if c != 'class']
X_train, y_train = df_train[FEAT_COLS], df_train['class']

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)

### Step 15: Evaluate on Test Split

In [ ]:
df_test = build_features('test')
X_test, y_test = df_test[FEAT_COLS], df_test['class']
y_pred = clf.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, digits=3))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred, labels=CLASSES))

### Step 16: Feature Importance

In [ ]:
imp = pd.Series(clf.feature_importances_, index=FEAT_COLS).sort_values()
imp.plot.barh(figsize=(7, 5))
plt.title('Morphology Feature Importance')
plt.tight_layout(); plt.show()